# 12 — Khoang phân cụm (M10)

**Mục đích.** Học, mổ và cấy ghép module phân cụm (M10): học thuật toán trên dữ liệu đồ chơi, mổ k-means của pipeline trên dữ liệu thật, rồi thử phương án thay thế (đợt 2). Đợt 1 chốt bộ đo tầng 1, cách lấy trạng thái RNG trước M10, và giao diện của khoang.

**Phạm vi.**
- Bối cảnh chính: cấu hình paper (`ft` r0, seed 4); `fz` seed 4 là bối cảnh phụ (PL-QD4).
- Câu hỏi liên quan: PL-Q1 (phân cụm thừa), PL-Q4 (nhãn tối thiểu cho lớp mới), PL-Q7 (so sánh phương pháp).
- Quy tắc: trên pool GC10 chỉ chạy K = 5 như pipeline. Phân cụm thừa (K' > 5) và các phương pháp khác chỉ chạy trên pool này sau khi đã tiền đăng ký (PL-E01, PL-E01b), để không nhìn trước dữ liệu. Muốn thử trên dữ liệu thật trước đó thì dùng mô phỏng khám phá trên lớp cơ sở, không dùng pool lớp mới.
- Số trong notebook chỉ để minh hoạ; số chính thức lấy từ `exp.py`.

In [1]:
# cell 2: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes")),
      "| features", Path("runs/inputs.txt").read_text().strip())


commit 3969b55 | code dirty True | features features/20261005_013042__gc10_fz_cls5first_img80pct/dino-vitb16


## Học — k-means và hiệu ứng đồng đều

**Định nghĩa.** k-means tìm K tâm $\mu_k$ để cực tiểu inertia (tổng bình phương trong cụm, within-cluster sum of squares):
$J = \sum_{k=1}^{K} \sum_{x \in C_k} \lVert x - \mu_k \rVert^2$.
Thuật toán Lloyd lặp hai bước: gán mỗi điểm vào tâm gần nhất, rồi đặt lại mỗi tâm bằng trung bình các điểm của cụm. $J$ không tăng sau mỗi bước, nên thuật toán dừng ở một cực tiểu địa phương (local minimum). Pipeline khởi tạo bằng k-means++, chạy 10 lần (`n_init`) và giữ lần có $J$ nhỏ nhất (Lloyd 1982; Arthur & Vassilvitskii 2007).

**"Có cụm riêng" nghĩa là gì.** Một nhóm có cụm riêng khi nó là lớp đông nhất trong một cụm (sở hữu cụm, như PL-E01). Không đòi cụm phải thuần: cụm của nhóm nhỏ có thể hút vài điểm ở đuôi xa của nhóm lớn.

**Hai sự kiện dùng trong ví dụ.**
- Gộp hai nhóm cỡ $n_1, n_2$ có tâm cách nhau $d$ làm $J$ tăng đúng $\frac{n_1 n_2}{n_1 + n_2} d^2$ (Ward 1963).
- Chẻ đôi $N(0, 1)$ tại 0: mỗi nửa có tâm $\pm\sqrt{2/\pi} \approx \pm 0.80$ và phương sai $1 - 2/\pi \approx 0.36$.

**Ví dụ (1 chiều, K = 2).** Nhóm lớn: 400 điểm theo $N(0, 1)$. Nhóm nhỏ: 15 điểm dồn quanh $x = \Delta$.
- A — tách nhóm nhỏ ra riêng: $J_A \approx 400 \times 1 = 400$.
- B — một phương án nuốt nhóm nhỏ: chẻ nhóm lớn tại 0, nhóm nhỏ nhập vào nửa phải (200 điểm): $J_B \approx 400 \times 0.36 + \frac{200 \cdot 15}{215}(\Delta - 0.8)^2 \approx 145 + 14(\Delta - 0.8)^2$.
- $\Delta = 4$: $J_B \approx 288 < 400$, nên nhóm nhỏ bị nuốt. $\Delta = 6$: $J_B \approx 523 > 400$, nên nhóm nhỏ có cụm riêng. Theo B, ngưỡng là $\Delta \approx 5.1$.
- B là ước lượng thô: k-means còn dời ranh giới chẻ để giảm $J$ thêm. Tính lại với ranh giới tối ưu (trên phân phối, không trên mẫu), ngưỡng nằm giữa 5 và 5.5.

**Tổng quát** (khi $m \ll n$; $\Delta$ đo bằng độ lệch chuẩn của nhóm lớn; theo phương án B): nhóm nhỏ có cụm riêng khi $\Delta \gtrsim 0.8 + \sqrt{\tfrac{2}{\pi} \cdot \tfrac{n}{m}}$. Hai nhóm càng lệch cỡ thì nhóm nhỏ phải càng xa mới được tách riêng. Đây là hiệu ứng đồng đều (uniform effect; Xiong, Wu & Chen 2009).

**Nối với GC10** [SUY LUẬN: mô hình 1 chiều, chỉ để minh hoạ]. silk/rolled = 420/17 ≈ 25, cho ngưỡng khoảng 4.8 lần độ trải của silk.

**Vì sao phân cụm thừa có thể giúp.** Chẻ nhóm lớn lần đầu giảm $J$ khoảng 0.64 cho mỗi điểm (từ 1 xuống 0.36). Chẻ thêm thì lợi ích giảm dần: với 3 cụm, $J$ mỗi điểm còn khoảng 0.19 (bộ lượng tử hoá tối ưu cho phân phối chuẩn; Max 1960). Khi tăng K', đến một lúc tách nhóm nhỏ ra có lợi hơn chẻ tiếp nhóm lớn. Đây là giả thuyết của PL-E01; cell 6 minh hoạ.

In [2]:
# cell 4 - toy check of the hand calculation: big group 400 ~ N(0, 1), small group 15 tight around x = delta; K = 2
# "own cluster" = the small group is the plurality in its cluster (PL-E01 ownership); purity is reported separately
import numpy as np
from sklearn.cluster import KMeans

rng = np.random.default_rng(0)
big = rng.normal(0.0, 1.0, size=(400, 1))
for delta in [3, 4, 5, 6, 7]:
    small = rng.normal(delta, 0.1, size=(15, 1))
    X = np.vstack([big, small])
    km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
    c = np.bincount(km.labels_[400:]).argmax()                       # cluster holding most of the small group
    n_small, n_big = (km.labels_[400:] == c).sum(), (km.labels_[:400] == c).sum()
    J_A = ((big - big.mean()) ** 2).sum() + ((small - small.mean()) ** 2).sum()   # option A, exact on this sample
    print(f"delta={delta}: cluster of the small group = {n_small} small + {n_big} big -> owns: {n_small > n_big}"
          f" | J={km.inertia_:.1f} | J_A={J_A:.1f}")

delta=3: cluster of the small group = 15 small + 166 big -> owns: False | J=204.5 | J_A=396.3
delta=4: cluster of the small group = 15 small + 120 big -> owns: False | J=275.4 | J_A=396.2
delta=5: cluster of the small group = 15 small + 115 big -> owns: False | J=369.6 | J_A=396.2
delta=6: cluster of the small group = 15 small + 1 big -> owns: True | J=394.7 | J_A=396.2
delta=7: cluster of the small group = 15 small + 0 big -> owns: True | J=396.3 | J_A=396.3


### Phân cụm thừa trên dữ liệu đồ chơi

Giữ $\Delta = 4$, mức mà với K = 2 nhóm nhỏ bị nuốt, rồi tăng K' = 2, 3, 4, 6.

Dự đoán với K' = 3, theo các số ở cell 3:
- Chẻ nhóm lớn làm hai và để nhóm nhỏ thành cụm riêng: $J \approx 400 \times 0.36 \approx 145$.
- Chẻ nhóm lớn làm ba rồi nuốt nhóm nhỏ vào phần bên phải (khoảng 108 điểm, tâm khoảng 1.22): $J \approx 400 \times 0.19 + \frac{108 \cdot 15}{123}(4 - 1.22)^2 \approx 76 + 102 = 178$.
- Vì 145 < 178, nhóm nhỏ có cụm riêng từ K' = 3 [SUY LUẬN: mô hình 1 chiều].

Cái giá: nhóm lớn bị chẻ thành nhiều cụm, giống silk ở GC10. Vì vậy cần một bước gộp các cụm con sau đó (PL-E02, nếu PL-E01 qua).

In [3]:
# cell 6 - toy over-clustering: keep delta = 4 (small group swallowed at K = 2), increase K'
small = np.random.default_rng(1).normal(4.0, 0.1, size=(15, 1))
X = np.vstack([big, small])
for k in [2, 3, 4, 6]:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    c = np.bincount(km.labels_[400:]).argmax()
    n_small, n_big = (km.labels_[400:] == c).sum(), (km.labels_[:400] == c).sum()
    print(f"K'={k}: cluster of the small group = {n_small} small + {n_big} big -> owns: {n_small > n_big}"
          f" | big group spread over {len(set(km.labels_[:400]))} clusters | J={km.inertia_:.1f}")

K'=2: cluster of the small group = 15 small + 120 big -> owns: False | big group spread over 2 clusters | J=274.6
K'=3: cluster of the small group = 15 small + 5 big -> owns: True | big group spread over 3 clusters | J=131.7
K'=4: cluster of the small group = 15 small + 3 big -> owns: True | big group spread over 4 clusters | J=74.4
K'=6: cluster of the small group = 15 small + 2 big -> owns: True | big group spread over 6 clusters | J=32.9


## Mổ — k-means trong pipeline và trạng thái RNG

`KMeansCluster` dựng `sklearn.cluster.KMeans(n_clusters=K, n_init=10, random_state=None)`. Với `random_state=None`, sklearn rút số ngẫu nhiên từ RNG toàn cục của numpy (`np.random`) đúng lúc gọi `fit`. Trạng thái RNG lúc đó phụ thuộc mọi bước chạy trước nó: M5 (loader) và M6 (PCA solver randomized), theo mục "Thứ tự chạy" trong README.

Muốn chạy lại M10 cho đúng, hoặc thử K' khác hay phương pháp khác từ cùng điểm xuất phát, phải chụp trạng thái RNG ngay lúc vào `fit`. Ở đây ta bọc `KMeansCluster.fit` từ bên ngoài để ghi `np.random.get_state()`. Lệnh này chỉ đọc, không rút số, nên không làm đổi kết quả của pipeline, và `replay.py` giữ nguyên.

Phép kiểm: fit lại bằng chính lớp `KMeansCluster` của pipeline, từ trạng thái đã chụp. Phân hoạch phải giống hệt phân hoạch của pipeline. Inertia được phép lệch ở bit cuối (xem sửa đổi I1 của PL-E01).

In [4]:
# cell 8 - Mổ: capture the global RNG state at the entry of KMeansCluster.fit (S3, option b: wrap from outside,
# replay.py unchanged), then refit with the pipeline's own class from that state; the partition must be identical
import contextlib, io
from pathlib import Path
from clustering.kmeans import KMeansCluster
from replay import replay

SEED, FT, K, OFF = 4, "gc10_ft_cls5first_img80pct_r0", 5, 5
fdir = sorted(Path("features").glob(f"*__{FT}"))
assert len(fdir) == 1, fdir
FEATURES = fdir[0] / "dino-vitb16-sl"

states, orig_fit = [], KMeansCluster.fit
def fit_recording(self, *args, **kwargs):
    states.append(np.random.get_state())            # read-only: does not consume the RNG
    return orig_fit(self, *args, **kwargs)
KMeansCluster.fit = fit_recording
buf = io.StringIO()
try:
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        r = replay(SEED, features=FEATURES, stop="cluster")     # S0 SVI still runs (~1 min)
finally:
    KMeansCluster.fit = orig_fit                     # always restore the class
assert len(states) == 1, len(states)
st = r["stages"][1]
X_pool, ids_pipe = st["train"]._x, np.asarray(st["raw"])

np.random.set_state(states[0])                       # same starting point as the pipeline's k-means
cl = KMeansCluster(num_classes=K, label_offset=OFF, n_init=10)
cl.fit(X_pool)
ids = np.asarray(cl.predict_raw(X_pool))
print("in :", X_pool.shape, "| K =", K, "| RNG states captured:", len(states))
print("same partition as the pipeline:", np.array_equal(ids, ids_pipe),
      "| inertia refit / pipeline:", cl.model.inertia_, "/", st["inertia"])

in : (748, 768) | K = 5 | RNG states captured: 1
same partition as the pipeline: True | inertia refit / pipeline: 82926.203125 / 82926.203125


## Bộ đo tầng 1 (chỉ dùng để chấm)

Đầu vào: nhãn thật `y` của ảnh pool (0–4 lớp cũ, 5–9 lớp mới; chỉ dùng để chấm) và id cụm `ids` (0 đến K' − 1). Bảng chéo $n_{ck}$ = số ảnh lớp $c$ trong cụm $k$; $n_c$, $n_k$ là tổng hàng, tổng cột.

- **Sở hữu cụm (ownership):** lớp $c$ sở hữu cụm $k$ nếu $c$ đông nhất trong $k$, tính cả 5 lớp cũ (như PL-E01). Hoà thì lấy lớp có id nhỏ hơn.
- **best(c):** $\max n_{ck}/n_c$ trên các cụm mà $c$ sở hữu; bằng 0 nếu $c$ không sở hữu cụm nào. Đây là đại lượng quyết định của PL-E01. **owned(c):** số cụm $c$ sở hữu.
- **Độ thuần (purity) của cụm:** $\max_c n_{ck}/n_k$ (Manning, Raghavan & Schütze 2008, §16.3); báo giá trị trung bình theo cụm.
- **Balanced accuracy sau ghép Hungarian một–một:** ghép mỗi lớp mới với một cụm khác nhau sao cho tổng số ảnh khớp lớn nhất (chỉ tính ảnh lớp mới; Kuhn 1955), rồi lấy trung bình recall của 5 lớp mới (Brodersen et al. 2010). So trực tiếp được với pipeline khi K' = 5; khi K' > 5, đại lượng này phạt các lớp bị chia ra nhiều cụm.
- **AMI** (adjusted mutual information) giữa `y` và `ids` trên cả pool: không phụ thuộc cách đặt tên cụm, đã trừ phần trùng khớp do may (Vinh, Epps & Bailey 2010).
- **Ảnh cũ trong cụm của lớp mới:** số ảnh lớp cũ nằm trong các cụm do lớp mới sở hữu. Nếu bộ lọc không loại, các ảnh này sẽ được học như ảnh lớp mới.
- **CV cỡ cụm:** $s/\bar{x}$ của cỡ các cụm ($s$ chia cho K' − 1); đo mức đồng đều mà k-means áp lên dữ liệu vốn lệch.

Tất cả tính trên pool (đánh giá chuyển nạp, transductive). Tập test chỉ dùng ở tầng 2, nên các số ở đây khác với recall trên test trong notebook 04.

In [5]:
# cell 10 - tier-1 scores of the pipeline's partition (ft r0, seed 4); score() lives in scripts/pl_cluster.py
import json
from pl_cluster import score

NAMES = [c.split("_")[0] for c in json.load(open("splits/gc10_split.json"))["classes"]]
y_pool = np.asarray(st["train"]._y).astype(int)
sc_ft = score(y_pool, ids_pipe, NAMES, n_old=OFF)
print("cluster sizes:", np.bincount(ids_pipe).tolist())
for k, v in sc_ft.items():
    print(f"{k:20s} {v:.3f}" if isinstance(v, float) else f"{k:20s} {v}")

cluster sizes: [95, 157, 272, 121, 103]
K                    5
ami                  0.346
size_cv              0.484
purity_mean          0.717
old_in_novel_owned   50
bal_acc_hungarian    0.317
owned_silk           3
best_silk            0.402
owned_inclusion      1
best_inclusion       0.341
owned_rolled         0
best_rolled          0.000
owned_crease         0
best_crease          0.000
owned_waist          1
best_waist           0.840


### Kiểm code của bộ đo trên `fz` seed 4

Bảng chéo của `fz` seed 4 đã có trong repo (`notes/improvement-2.vi.md`, Q1): cỡ cụm 267, 163, 123, 105, 90. Từ bảng đó, tính tay được:
- silk sở hữu 3 cụm, best = 175/420; inclusion sở hữu 1 cụm, best = 49/135; waist sở hữu 1 cụm, best = 92/94; crease và rolled không sở hữu cụm nào.
- Balanced accuracy sau Hungarian khoảng 35.2%. Con số này có thể nhích lên tối đa khoảng 37.6% nếu 3 ảnh chưa rõ lớp nằm trong hai cụm gần như toàn silk là crease hoặc rolled.

Code ra đúng các số trên thì code bộ đo đúng.

In [6]:
# cell 12 - check score() on fz seed 4 against the crosstab already in the repo (notes/improvement-2.vi.md, Q1)
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    r_fz = replay(SEED, stop="cluster")                    # default features = runs/inputs.txt (fz)
st_fz = r_fz["stages"][1]
ids_fz = np.asarray(st_fz["raw"])
sizes_fz = sorted(np.bincount(ids_fz).tolist(), reverse=True)
print("features:", r_fz["args"].data_dir, "| cluster sizes:", sizes_fz)
assert sizes_fz == [267, 163, 123, 105, 90], sizes_fz
sc_fz = score(np.asarray(st_fz["train"]._y).astype(int), ids_fz, NAMES, n_old=OFF)
for k, v in sc_fz.items():
    print(f"{k:20s} {v:.3f}" if isinstance(v, float) else f"{k:20s} {v}")

features: features/20261005_013042__gc10_fz_cls5first_img80pct | cluster sizes: [267, 163, 123, 105, 90]
K                    5
ami                  0.378
size_cv              0.475
purity_mean          0.748
old_in_novel_owned   50
bal_acc_hungarian    0.358
owned_silk           3
best_silk            0.417
owned_inclusion      1
best_inclusion       0.363
owned_rolled         0
best_rolled          0.000
owned_crease         0
best_crease          0.000
owned_waist          1
best_waist           0.979


## Quyết định (đợt 1)

- **Bộ đo tầng 1:** như cell 9; code ở `scripts/pl_cluster.py` (`score`). Đã kiểm: ra đúng các số tính tay từ bảng chéo, trên cả `ft` và `fz`. Với `fz`, balanced accuracy 0.358 ≈ 0.352 + (1/32)/5, tức đúng một ảnh crease rơi vào cụm silk được ghép với crease.
- **Trạng thái RNG trước M10 (S3):** bọc `KMeansCluster.fit` từ bên ngoài để chụp `np.random.get_state()`; `replay.py` giữ nguyên. Fit lại từ trạng thái đó cho đúng phân hoạch của pipeline (cell 8).
- **Giao diện của khoang:** `f(X_pool, K, rng_state, X_lab=None, y_lab=None) -> ids`, id cụm từ 0 đến K − 1. `X_lab`, `y_lab` (dữ liệu có nhãn của S0) là tuỳ chọn, để sau này thêm phương pháp bán giám sát (ví dụ semi-supervised k-means; Vaze et al. 2022) mà không phải đổi giao diện. Phương pháp gán nhãn nhiễu (−1) phải đổi nhãn đó về id cụm hợp lệ trước khi chấm.
- **Không nhìn trước dữ liệu:** K' > 5 và các phương pháp khác chỉ chạy trên pool GC10 sau khi đã tiền đăng ký.
- **Tiếp theo:** script tầng 1 chạy PL-E01 (`fz`, đã tiền đăng ký) và PL-E01b (`ft`, tiền đăng ký trước khi chạy). Đợt 2 của notebook (Cấy) đọc kết quả của chúng.

## Summary (EN)

Batch 1 (learn + dissect); illustrative numbers, single seed (seed 4).

1. Toy model: with a 400:15 size ratio, k-means with K = 2 absorbs the small group unless it lies about 5–6 standard deviations away; at a distance of 4, over-clustering (K' ≥ 3) gives the small group its own cluster, at the cost of splitting the large group (whose distortion follows the optimal Gaussian quantizer; Max 1960).
2. The pipeline's k-means draws from the global NumPy RNG at fit time; capturing that state at fit entry, without changing the pipeline, reproduces the pipeline's partition exactly (paper setting, FT seed 0, seed 4).
3. At K = 5 on the S1 pool, crease and rolled_pit own no cluster in both feature settings, silk_spot owns three clusters, and all 50 old pool images fall in novel-owned clusters. Balanced accuracy after one-to-one Hungarian matching is 31.7% with fine-tuned features (run `gc10_ft_cls5first_img80pct_r0_km_evhung_d64_s4`) and 35.8% with frozen features (run `gc10_frozen_km_evhung_d64_head_s4`).